# Раздел 8 · Multitenancy: изоляция клиентов в SaaS

In [1]:
%pip install qdrant-client sentence-transformers ipykernel --quiet


/home/dimon/vscode/examples_qdrant/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    KeywordIndexParams, Filter, FieldCondition, MatchValue
)

print("OK: библиотеки импортированы")


/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: библиотеки импортированы


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [3]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(
    url="http://localhost:6336",
    api_key="Fucking_password_1234567890",   # значение из stepik/.env
)
print(client.get_collections())


qdrant_lecture
collections=[CollectionDescription(name='day1_semantic_search'), CollectionDescription(name='exercise1_books')]


/tmp/ipykernel_121897/1508062746.py:21: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


In [4]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37597.59it/s]


Размерность эмбеддинга: 384 | книг в подборке: 87


In [5]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


OK: exercise1_books - 87 точек


## Multitenancy

Тоже продолжение `BOOKS_COLLECTION` — просто добавляем новое поле `tenant_id` через
`set_payload` поверх уже загруженных 87 точек и специальный индекс `is_tenant` поверх него.

### Пример — разметка тенантов и is_tenant-индекс

Понарошку делим 87 произведений на два "читальных зала" по литературному направлению:

In [6]:
for tenant in ("романтизм", "реализм"):
    # переиспользуем уже знакомый set_payload - просто размечаем каждую точку её тенантом
    ids = [b["id"] for b in BOOKS if b["movement"] == tenant]
    client.set_payload(collection_name=BOOKS_COLLECTION, payload={"tenant_id": tenant}, points=ids)

# is_tenant=True - подсказка Qdrant, что по этому полю коллекция физически партиционируется
# (данные одного тенанта хранятся рядом на диске) - именно так реализуется multitenancy в одной коллекции
client.create_payload_index(
    BOOKS_COLLECTION, field_name="tenant_id",
    field_schema=KeywordIndexParams(type="keyword", is_tenant=True),
)

for tenant in ("романтизм", "реализм"):
    count = client.count(
        BOOKS_COLLECTION,
        count_filter=Filter(must=[FieldCondition(key="tenant_id", match=MatchValue(value=tenant))]),
    ).count
    print(tenant, "->", count, "произведений")


романтизм -> 16 произведений
реализм -> 71 произведений


Реальный результат: `романтизм -> 16`, `реализм -> 71` (сумма — все 87).

## Задание 1

Найдите топ-5 по запросу **"любовь и свобода среди дикой природы"**, но только среди произведений
`tenant_id="романтизм"` — так же, как в разделе про фильтрацию вы уже фильтровали по `genre`/`year`, только поле
теперь другое. Сохраните в `hits_ex1`.

**Используйте:** `client.query_points(...)`, `query_filter=Filter(must=[FieldCondition(key="tenant_id", ...)])`.

In [7]:
# ВАШ КОД ЗДЕСЬ
hits_ex1 = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("любовь и свобода среди дикой природы"),
    query_filter=Filter(
        must=[FieldCondition(key="tenant_id", match=MatchValue(value="романтизм"))]
    ),
    limit=5
)

In [8]:
hits_ex1

QueryResponse(points=[ScoredPoint(id=66, version=8, score=0.8055595, payload={'title': 'Демон', 'author': 'Михаил Лермонтов', 'year': 1839, 'genre': 'поэма', 'movement': 'романтизм', 'pages': 35, 'in_school_curriculum': True, 'text': 'Гордый и одинокий падший дух Демон, уставший от вечного зла и презрения ко всему сущему, пролетая над Кавказом видит юную грузинскую княжну Тамару и впервые за долгие века испытывает искреннее чувство любви и надежду на возрождение. Демон является Тамаре, добивается её любви и клянётся в раскаянии, но поцелуй его убивает девушку. Ангел уносит душу Тамары на небо, а обманутый в надежде на искупление Демон остаётся вечно одиноким среди холодного бесстрастного мира.', 'tenant_id': 'романтизм'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=63, version=8, score=0.80449474, payload={'title': 'Цыганы', 'author': 'Александр Пушкин', 'year': 1827, 'genre': 'поэма', 'movement': 'романтизм', 'pages': 25, 'in_school_curriculum': False, 'text': 'Разо

In [9]:
def check_ex1(hits_ex1):
    assert len(hits_ex1) > 0, "поиск вернул пустой результат"
    for h in hits_ex1:
        assert h.payload["tenant_id"] == "романтизм", (
            "утечка между тенантами: точка id={} принадлежит tenant_id={}".format(h.id, h.payload["tenant_id"])
        )
    print("OK: все {} результатов из романтизм, id={}".format(len(hits_ex1), [h.id for h in hits_ex1]))
    print("ОТВЕТ ДЛЯ STEPIK:", hits_ex1[0].id)


check_ex1(hits_ex1)


TypeError: object of type 'QueryResponse' has no len()